# Part 4a / 4b — From cooperative planner to a two-firm game

### Where one objective function stops being enough

Parts 1–3b all had **one** objective, which means one decision maker. This notebook takes the
first two steps toward rivalry:

| | Model | Method | What it is |
|---|---|---|---|
| **4a** | one planner, two weighted regional costs | MILP, sweep the weight | **Not a game.** A cooperative bound and a Pareto frontier |
| **4b** | two firms, **fixed** price | iterative best response | The first genuine strategic interaction |

4c (endogenous price / Cournot), 4d (Stackelberg via KKT) and 4e (policy instruments) follow in
later notebooks.

### Institutional split: government constrains, firms optimize

- **Government** sets non-market constraints — local content minimums, tariffs, quotas. Exogenous
  and swept, never chosen by the model.
- **Firms** are profit maximizers operating within those constraints. One firm per region.

This is what justifies per-tier demand minimums as policy rather than as a modelling trick, and it
avoids a trilevel structure where governments and firms both optimize.

### Structural change from Part 3b: vertical integration

In Part 3b any stage could source from either region. That is fine for a single planner, but in a
game it would require an internal transfer price between rival firms at every stage — a modelling
problem in its own right. So from here on:

- each firm owns a **vertically integrated chain** inside its own region (MINE → PROC → MFG)
- firms **compete downstream**, delivering finished product into either region's demand market
- cross-region delivery pays a transport premium (2.4 vs 0.5)

### An asymmetric instance, deliberately

Symmetric regions produce symmetric results and teach nothing about entry. Here:

| | R1 — incumbent | R2 — entrant |
|---|---|---|
| Build cost | higher | ~9% lower |
| Operating cost | lower | higher |
| Legacy capacity | larger | smaller |
| Accumulated production experience | **2,600** | 500 |

R1 begins deep into its learning curve; R2 is cheaper to build but starts inexperienced. The
question is whether R2 can buy its way down the curve before R1's cost advantage locks it out.

### Both learning channels are retained

Capacity→capex (SOS2) and production→opex (lagged tiers) are both kept, even though Part 3b showed
production learning does not move the plan under cost minimization. In the game, production stops
being pinned by demand — so the channel finally has a lever, and the effect should be small but
real at this model size.

## 1. Setup

In [ ]:
!pip install gurobipy --quiet
import math
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})
print("gurobipy", gp.gurobi.version())

## 2. Sets, time, technology

Unchanged in structure from Part 3b: variable-length periods, $\omega_p$ for money and $L_p$ for
physical accumulation, CRF-annualised capex, vintage-indexed yield.

In [ ]:
REGIONS = ['R1', 'R2']
STAGES  = ['MINE', 'PROC', 'MFG']

# ---------------- TIME ----------------
BLOCKS = [(6, 1), (4, 3), (2, 5), (1, 9)]
LEN, START = [], []
_y = 1
for _c, _L in BLOCKS:
    for _ in range(_c):
        LEN.append(_L); START.append(_y); _y += _L
P = list(range(len(LEN)))
HORIZON = _y - 1
YEARS = {p: list(range(START[p], START[p] + LEN[p])) for p in P}
DR = 0.05
OMEGA = {p: sum(1/(1+DR)**t for t in YEARS[p]) for p in P}
YEAR_TO_P = {t: p for p in P for t in YEARS[p]}
REPORT_UNTIL = 28

# ---------------- TECH ----------------
LIFE = 25
LEAD = {'MINE': 1, 'PROC': 2, 'MFG': 2}
CAP_MIN, CAP_MAX = 60.0, 260.0
CRF = DR*(1+DR)**LIFE/((1+DR)**LIFE - 1)
ONLINE = {(s, p): START[p] + LEAD[s] for s in STAGES for p in P}
MU = {(s, v): CRF*sum(1/(1+DR)**t for t in range(ONLINE[s, v], ONLINE[s, v]+LIFE)
                      if t <= HORIZON) for s in STAGES for v in P}

## 3. The asymmetric instance

`EXPERIENCE0` is the new parameter and it carries the whole entry story: R1 starts with 2,600 units
of accumulated production, R2 with 500. Since opex learning tiers depend on cumulative production,
R1 begins several tiers ahead.

In [ ]:
# ---------------- ASYMMETRIC INSTANCE ----------------
# R1 = incumbent upstream processor with accumulated experience.
# R2 = entrant, cheaper to build, trying to move downstream.
FIXED = {('MINE','R1'):900.,('PROC','R1'):1500.,('MFG','R1'):1300.,
         ('MINE','R2'):820.,('PROC','R2'):1350.,('MFG','R2'):1180.}
UNIT  = {('MINE','R1'):7.0,('PROC','R1'):11.0,('MFG','R1'):9.5,
         ('MINE','R2'):6.4,('PROC','R2'):10.0,('MFG','R2'):8.7}
OPEX  = {('MINE','R1'):1.2,('PROC','R1'):2.0,('MFG','R1'):2.4,
         ('MINE','R2'):1.35,('PROC','R2'):2.2,('MFG','R2'):2.6}

LEGACY_CAP = {('MINE','R1'):230,('PROC','R1'):205,('MFG','R1'):155,
              ('MINE','R2'):165,('PROC','R2'):125,('MFG','R2'):100}
LEGACY_RET = {('MINE','R1'):11,('PROC','R1'):14,('MFG','R1'):18,
              ('MINE','R2'):9, ('PROC','R2'):16,('MFG','R2'):22}
LEGACY_BYR = -8
# incumbent starts with accumulated production experience
EXPERIENCE0 = {'R1': 2600.0, 'R2': 500.0}

In [ ]:
# ---------------- EFFICIENCY (yield) ----------------
ETA_CEIL = {'MINE':0.92,'PROC':0.95,'MFG':0.93}
ETA_BASE = {'MINE':0.86,'PROC':0.80,'MFG':0.78}
ALPHA    = {'MINE':0.0,'PROC':0.030,'MFG':0.025}
BETA     = {'MINE':0.0,'PROC':0.010,'MFG':0.008}
DELTA_BAR= {'MINE':0.02,'PROC':0.05,'MFG':0.05}
ETA_FLOOR= 0.60
VINTAGES = [-1] + P
BYEAR = {v: (LEGACY_BYR if v == -1 else START[v]) for v in VINTAGES}
ETA = {}
for s in STAGES:
    for v in VINTAGES:
        fr = ETA_CEIL[s] - (ETA_CEIL[s]-ETA_BASE[s])*(1-ALPHA[s])**(BYEAR[v]-1)
        fr = max(ETA_FLOOR, min(fr, ETA_CEIL[s]))
        for p in P:
            age = max(0, START[p]-BYEAR[v])
            aged = ETA_CEIL[s] - (ETA_CEIL[s]-fr)*(1-BETA[s])**age
            ETA[s, v, p] = max(ETA_FLOOR, min(fr+DELTA_BAR[s], aged))

In [ ]:
# ---------------- DEMAND & MARKET ----------------
DEMAND = {}
for r, base, g in [('R1', 100.0, 0.008), ('R2', 75.0, 0.026)]:
    for p in P:
        DEMAND[r, p] = sum(base*(1+g)**(t-1) for t in YEARS[p])/LEN[p]
TRANSPORT = {(rf, rt): (0.5 if rf == rt else 2.4) for rf in REGIONS for rt in REGIONS}
PRICE_FIXED = 12.0
PEN_SHORT, PEN_DISPOSE = 90.0, 12.0

In [ ]:
# ---------------- LEARNING ----------------
LEARN_STAGES = ['PROC', 'MFG']
LR_CAPEX, Q_START, Q_ADD, CAPEX_FLOOR, NBP = 0.15, 300.0, 700.0, 0.60, 9
_bc = -math.log2(1-LR_CAPEX)
K = list(range(NBP))
QBP = [Q_START + Q_ADD*k/(NBP-1) for k in K]

def _cap_unit_mult(q):
    return max(CAPEX_FLOOR, (q/Q_START)**(-_bc))

def _cap_cum_mult(q, n=400):
    if q <= Q_START:
        return 0.0
    h = (q-Q_START)/n
    return sum(0.5*(_cap_unit_mult(Q_START+i*h)+_cap_unit_mult(Q_START+(i+1)*h))*h
               for i in range(n))
CBP = [_cap_cum_mult(q) for q in QBP]

LR_OPEX, OPEX_FLOOR, LAG_YEARS, N_TIERS = 0.18, 0.65, 3, 3
TIER_Q, TIER_M = {}, {}

def set_tiers(top_by_region):
    for r in REGIONS:
        top = max(top_by_region[r], 1.0)
        q1 = top/8.0
        TIER_Q[r] = [q1*2**j for j in range(N_TIERS-1)]
        TIER_M[r] = [max(OPEX_FLOOR, (1-LR_OPEX)**j) for j in range(N_TIERS)]

ACTIVE = {r: [(s, v, p) for s in STAGES for v in VINTAGES for p in P
              if (v == -1 and START[p] <= LEGACY_RET[s, r])
              or (v >= 0 and ONLINE[s, v] <= START[p] <= ONLINE[s, v]+LIFE-1)]
          for r in REGIONS}
VIN = {(r, s, p): [v for (ss, v, pp) in ACTIVE[r] if (ss, pp) == (s, p)]
       for r in REGIONS for s in STAGES for p in P}
BUILD = {r: [(s, v) for s in STAGES for v in P if ONLINE[s, v] <= HORIZON]
         for r in REGIONS}

## 4. One region's chain

`add_region` attaches a single firm's vertically integrated chain to a model and returns its cost
and revenue expressions. It is used **three ways**:

- twice in the same model → the cooperative planner of 4a
- once alone → a firm's best-response problem in 4b
- later, as the follower's block in the bilevel model of 4d

Keeping the chain in one place is what makes those three models share verified code. Inside, the
constraints are the same flat `addConstrs` style as Part 3.

Note `cum[p]` starts from `EXPERIENCE0[r]`, and note the final balance: manufacturing output must
equal **sales plus disposal**, which is what allows overproduction.

In [ ]:
def add_region(m, r, learning='both'):
    """Attach one region's vertically-integrated chain to model m. Returns handles."""
    b = m.addVars(BUILD[r], vtype=GRB.BINARY, name=f'b_{r}')
    c = m.addVars(BUILD[r], lb=0.0, ub=CAP_MAX, name=f'c_{r}')
    x = m.addVars(ACTIVE[r], lb=0.0, name=f'x_{r}')
    f_mp = m.addVars(P, lb=0.0, name=f'fmp_{r}')
    f_pf = m.addVars(P, lb=0.0, name=f'fpf_{r}')
    sale = m.addVars(REGIONS, P, lb=0.0, name=f'sale_{r}')
    disp = m.addVars(P, lb=0.0, name=f'disp_{r}')

    m.addConstrs((c[s, v] <= CAP_MAX*b[s, v] for (s, v) in BUILD[r]), name=f'su_{r}')
    m.addConstrs((c[s, v] >= CAP_MIN*b[s, v] for (s, v) in BUILD[r]), name=f'sl_{r}')
    m.addConstrs((x[s, v, p] <= (LEGACY_CAP[s, r] if v == -1 else c[s, v])
                  for (s, v, p) in ACTIVE[r]), name=f'cap_{r}')
    m.addConstrs((gp.quicksum(ETA['MINE', v, p]*x['MINE', v, p]
                              for v in VIN[r, 'MINE', p]) == f_mp[p] for p in P),
                 name=f'mine_{r}')
    m.addConstrs((f_mp[p] == gp.quicksum(x['PROC', v, p] for v in VIN[r, 'PROC', p])
                  for p in P), name=f'pin_{r}')
    m.addConstrs((gp.quicksum(ETA['PROC', v, p]*x['PROC', v, p]
                              for v in VIN[r, 'PROC', p]) == f_pf[p] for p in P),
                 name=f'pout_{r}')
    m.addConstrs((f_pf[p] == gp.quicksum(x['MFG', v, p] for v in VIN[r, 'MFG', p])
                  for p in P), name=f'min_{r}')
    m.addConstrs((gp.quicksum(ETA['MFG', v, p]*x['MFG', v, p]
                              for v in VIN[r, 'MFG', p])
                  == sale.sum('*', p) + disp[p] for p in P), name=f'mout_{r}')

    # cumulative production (undiscounted), regional scope, with initial experience
    cum = m.addVars(P, lb=0.0, ub=3*CAP_MAX*HORIZON + EXPERIENCE0[r], name=f'cum_{r}')
    m.addConstrs((cum[p] == EXPERIENCE0[r] +
                  gp.quicksum(LEN[q]*x['MFG', v, q] for q in P if q <= p
                              for v in VIN[r, 'MFG', q]) for p in P), name=f'cp_{r}')

    capex = gp.quicksum(MU[s, v]*FIXED[s, r]*b[s, v] for (s, v) in BUILD[r]) \
          + gp.quicksum(MU[s, v]*UNIT[s, r]*c[s, v]
                        for (s, v) in BUILD[r] if s not in LEARN_STAGES)
    if learning in ('capacity', 'both'):
        Q = m.addVars(P, lb=Q_START, ub=Q_START+Q_ADD, name=f'Q_{r}')
        Cc = m.addVars(P, lb=0.0, name=f'C_{r}')
        lam = m.addVars(P, K, lb=0.0, ub=1.0, name=f'lam_{r}')
        m.addConstrs((lam.sum(p, '*') == 1 for p in P), name=f'sc_{r}')
        m.addConstrs((Q[p] == gp.quicksum(QBP[k]*lam[p, k] for k in K) for p in P),
                     name=f'sQ_{r}')
        m.addConstrs((Cc[p] == gp.quicksum(CBP[k]*lam[p, k] for k in K) for p in P),
                     name=f'sC_{r}')
        m.addConstrs((Q[p] == Q_START + gp.quicksum(c[s, v] for (s, v) in BUILD[r]
                                                    if s in LEARN_STAGES and v <= p)
                      for p in P), name=f'cc_{r}')
        for p in P:
            m.addSOS(GRB.SOS_TYPE2, [lam[p, k] for k in K])
        rate = sum(UNIT[s, r] for s in LEARN_STAGES)/len(LEARN_STAGES)
        capex += gp.quicksum(MU['PROC', p]*rate*(Cc[p]-(Cc[p-1] if p > 0 else 0.0))
                             for p in P)
    else:
        capex += gp.quicksum(MU[s, v]*UNIT[s, r]*c[s, v]
                             for (s, v) in BUILD[r] if s in LEARN_STAGES)

    if learning in ('production', 'both') and TIER_Q:
        J = list(range(N_TIERS))
        z = m.addVars(P, J, vtype=GRB.BINARY, name=f'z_{r}')
        m.addConstrs((z.sum(p, '*') == 1 for p in P), name=f'ot_{r}')
        LAGP = {p: YEAR_TO_P[max(1, START[p]-LAG_YEARS)] for p in P}
        BIGQ = 3*CAP_MAX*HORIZON + EXPERIENCE0[r]
        m.addConstrs((cum[LAGP[p]] >= TIER_Q[r][j-1] - BIGQ*(1-z[p, j])
                      for p in P for j in J if j > 0), name=f'tf_{r}')
        m.addConstrs((cum[LAGP[p]] <= TIER_Q[r][j] + BIGQ*(1-z[p, j])
                      for p in P for j in J if j < N_TIERS-1), name=f'tc_{r}')
        ts = m.addVars(STAGES, P, J, lb=0.0, name=f'ts_{r}')
        m.addConstrs((ts.sum(s, p, '*') == gp.quicksum(x[s, v, p] for v in VIN[r, s, p])
                      for s in STAGES for p in P), name=f'tss_{r}')
        m.addConstrs((ts[s, p, j] <= 3*CAP_MAX*z[p, j]
                      for s in STAGES for p in P for j in J), name=f'tl_{r}')
        opex = gp.quicksum(OMEGA[p]*OPEX[s, r]*TIER_M[r][j]*ts[s, p, j]
                           for s in STAGES for p in P for j in J)
    else:
        z = None
        opex = gp.quicksum(OMEGA[p]*OPEX[s, r]*x[s, v, p] for (s, v, p) in ACTIVE[r])

    trans = gp.quicksum(OMEGA[p]*TRANSPORT[r, rt]*sale[rt, p] for rt in REGIONS for p in P)
    dcost = gp.quicksum(OMEGA[p]*PEN_DISPOSE*disp[p] for p in P)
    revenue = gp.quicksum(OMEGA[p]*PRICE_FIXED*sale[rt, p] for rt in REGIONS for p in P)
    return dict(b=b, c=c, x=x, sale=sale, disp=disp, cum=cum, z=z,
                capex=capex, opex=opex, trans=trans, dcost=dcost, revenue=revenue,
                cost=capex+opex+trans+dcost)

## 5. Model 4a — the cooperative planner

$$\min \;\; w\, \text{Cost}_{R1} + (1-w)\, \text{Cost}_{R2} + \pi^{short}\!\sum u$$

subject to both firms' chains and a **shared** demand constraint: combined deliveries must cover
each market.

**This is not a game and should not be read as one.** One objective function means one decision
maker; the weight $w$ is a planner's relative valuation of the two regions, not a bargaining
outcome. Sweeping $w$ traces the Pareto frontier of achievable cost pairs, which is the right
cooperative benchmark against which to measure the cost of rivalry later.

In [ ]:
def solve_planner(w1=0.5, learning='both', mipgap=0.005, quiet=True):
    m = gp.Model(); m.Params.OutputFlag = 0 if quiet else 1; m.Params.MIPGap = mipgap
    H = {r: add_region(m, r, learning) for r in REGIONS}
    short = m.addVars(REGIONS, P, lb=0.0, name='short')
    m.addConstrs((gp.quicksum(H[r]['sale'][rt, p] for r in REGIONS) + short[rt, p]
                  >= DEMAND[rt, p] for rt in REGIONS for p in P), name='demand')
    pen = gp.quicksum(OMEGA[p]*PEN_SHORT*short[rt, p] for rt in REGIONS for p in P)
    m.setObjective(w1*H['R1']['cost'] + (1-w1)*H['R2']['cost'] + pen, GRB.MINIMIZE)
    m.optimize()
    m._H, m._short, m._pen = H, short, pen
    return m

### Calibrating the opex tiers

Same discipline as Part 3b: solve without production learning, read the cumulative production that
actually occurs, and place thresholds across that range as doublings.

In [ ]:
m0 = solve_planner(0.5, learning='capacity')
top = {r: m0._H[r]['cum'][P[-1]].X for r in REGIONS}
set_tiers(top)
print(f"planner objective without production learning: {m0.ObjVal:.1f}")
print("cumulative MFG production by region:", {r: round(top[r], 1) for r in REGIONS})
print("tier thresholds :", {r: [round(q, 1) for q in TIER_Q[r]] for r in REGIONS})
print("tier multipliers:", {r: [round(x, 3) for x in TIER_M[r]] for r in REGIONS})

### The Pareto frontier

In [ ]:
rows = []
for w1 in [0.1, 0.3, 0.5, 0.7, 0.9]:
    m = solve_planner(w1, learning='both')
    H = m._H
    rows.append(dict(weight_R1=w1, weighted_obj=round(m.ObjVal, 1),
                     cost_R1=round(H['R1']['cost'].getValue(), 1),
                     cost_R2=round(H['R2']['cost'].getValue(), 1),
                     builds_R1=sum(1 for k in H['R1']['b'] if H['R1']['b'][k].X > 0.5),
                     builds_R2=sum(1 for k in H['R2']['b'] if H['R2']['b'][k].X > 0.5),
                     shortfall=round(sum(m._short[rt, p].X
                                         for rt in REGIONS for p in P), 2)))
dfA = pd.DataFrame(rows); dfA

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(dfA.cost_R1, dfA.cost_R2, 'o-', lw=2.5, ms=11, color='#2471a3')
for _, r_ in dfA.iterrows():
    ax[0].annotate(f"w={r_.weight_R1:.1f}", (r_.cost_R1, r_.cost_R2),
                   textcoords='offset points', xytext=(8, 8), fontsize=10)
ax[0].set_xlabel('cost borne by R1'); ax[0].set_ylabel('cost borne by R2')
ax[0].set_title('Pareto frontier (cooperative planner)')
w = 0.35
ax[1].bar(dfA.weight_R1 - w/2*0.1, dfA.builds_R1, 0.05, label='R1 builds', color='#2471a3')
ax[1].bar(dfA.weight_R1 + w/2*0.1, dfA.builds_R2, 0.05, label='R2 builds', color='#d68910')
ax[1].set_xlabel('weight on R1'); ax[1].set_ylabel('facilities built'); ax[1].legend()
ax[1].set_title('Who builds, as the planner shifts weight')
plt.tight_layout(); plt.show()

The frontier is sharply **non-convex and nearly bang-bang**: the planner loads essentially all
capacity onto whichever region it weights less, because that region's cost is discounted in the
objective. Between $w = 0.3$ and $w = 0.7$ the build allocation flips 6/3 → 3/3 → 3/6.

That shape is a direct consequence of lumpy investment — you cannot build 0.4 of a facility, so the
frontier is a set of discrete points rather than a smooth curve. It is also why a weighted-sum
scalarisation is a weak tool for this problem: whole regions of the frontier are unreachable by any
weight. An $\varepsilon$-constraint formulation (minimise R1 cost subject to R2 cost $\le$ budget)
would trace it more faithfully and is worth trying.

Note the shortfall column rising at extreme weights: when the planner nearly ignores one region it
starts letting demand go unserved rather than building there.

## 6. Model 4b — two firms, fixed price

Now each region is a **profit-maximising firm**:

$$\max_{\text{firm } r} \;\; \underbrace{\sum_p \omega_p\, \bar{p} \sum_{rt} \text{sale}_{r,rt,p}}_{\text{revenue}}
\;-\; \underbrace{\text{Cost}_r}_{\text{capex + opex + transport + disposal}}$$

At a **fixed** price, nobody buys more than the market wants, so the firm's sales are capped by
**residual demand** — what the rival has left unserved:

$$\text{sale}_{r,rt,p} \;\le\; \max\{0,\; D_{rt,p} - \overline{\text{sale}}_{-r,rt,p}\}$$

That single constraint is the entire channel of rivalry here, and it makes the structure a **race
for market share**: at any price above marginal cost both firms want the whole market, and whoever
commits capacity first captures it.

**Best-response iteration**: fix the rival's schedule, solve one firm's MILP, swap, repeat.

In [ ]:
def best_response(r, rival_sales, learning='both', mipgap=0.005):
    """Firm r maximises profit given the rival's delivery schedule."""
    m = gp.Model(); m.Params.OutputFlag = 0; m.Params.MIPGap = mipgap
    h = add_region(m, r, learning)
    # at a fixed price nobody buys more than residual demand
    m.addConstrs((h['sale'][rt, p] <= max(0.0, DEMAND[rt, p] - rival_sales.get((rt, p), 0.0))
                  for rt in REGIONS for p in P), name='residual')
    m.setObjective(h['revenue'] - h['cost'], GRB.MAXIMIZE)
    m.optimize()
    m._h = h
    return m

In [ ]:
def profile(m):
    h = m._h
    return {(rt, p): h['sale'][rt, p].X for rt in REGIONS for p in P}


def plan_key(m):
    h = m._h
    return tuple(sorted((s, v) for (s, v) in h['b'] if h['b'][s, v].X > 0.5))

### Termination — three exits, and the third is the interesting one

Best-response iteration on a game with **indivisible** investment need not converge. Three
distinct outcomes must be distinguished:

1. **Converged** — the strategy profile repeats immediately. A fixed point: a pure-strategy Nash
   equilibrium of the discretised game.
2. **Cycle** — the profile repeats after $k \ge 2$ rounds. **No pure-strategy equilibrium was
   found**, and the cycle itself is the result: each firm builds only if the other does not. This is
   real economics, not a bug, and suppressing it would be the actual error.
3. **Iteration cap** — report non-convergence honestly.

Convergence is tested on **strategies, not objective values**: profits can be nearly identical
while the underlying build plans oscillate between genuinely different configurations.

In [ ]:
def iterate_best_response(learning='both', first='R1', max_iter=12, tol=1e-4, verbose=True):
    sales = {r: {(rt, p): 0.0 for rt in REGIONS for p in P} for r in REGIONS}
    plans, hist, log = {}, [], []
    order = [first, 'R2' if first == 'R1' else 'R1']
    for it in range(max_iter):
        for r in order:
            other = 'R2' if r == 'R1' else 'R1'
            m = best_response(r, sales[other], learning=learning)
            if m.SolCount == 0:
                return dict(status='INFEASIBLE', iters=it, log=log)
            sales[r] = profile(m)
            plans[r] = plan_key(m)
            log.append(dict(iter=it, firm=r, profit=m.ObjVal,
                            builds=len(plans[r]),
                            total_sales=sum(sales[r].values())))
        state = (plans.get('R1'), plans.get('R2'))
        if state in hist:
            # a state repeating IMMEDIATELY is a fixed point; repeating after k
            # rounds is a genuine k-cycle with no pure-strategy equilibrium found
            clen = len(hist) - hist.index(state)
            return dict(status=('CONVERGED' if clen == 1 else 'CYCLE'),
                        cycle_len=clen, iters=it+1, log=log,
                        plans=plans, sales=sales, hist=hist)
        hist.append(state)
    return dict(status='MAX_ITER', iters=max_iter, log=log, plans=plans, sales=sales)

### Does it converge, and does order matter?

In [ ]:
rows = []
for first in ['R1', 'R2']:
    res = iterate_best_response(first=first, max_iter=10)
    last = {L['firm']: L for L in res['log'][-2:]}
    rows.append(dict(first_mover=first, status=res['status'],
                     repeat_length=res.get('cycle_len'), iterations=res['iters'],
                     profit_R1=round(last['R1']['profit'], 1),
                     profit_R2=round(last['R2']['profit'], 1),
                     sales_R1=round(last['R1']['total_sales'], 1),
                     sales_R2=round(last['R2']['total_sales'], 1)))
dfB = pd.DataFrame(rows); dfB

Both orders reach a **fixed point** (repeat length 1), so a pure-strategy equilibrium exists in this
instance — but they reach **different** ones:

| First mover | R1 profit | R2 profit | R1 sales | R2 sales |
|---|---|---|---|---|
| R1 | **7,613** | 2,779 | 1,652 | 987 |
| R2 | 5,894 | **3,722** | 1,361 | 1,304 |

**Moving first is worth roughly 29% of profit to R1 and 34% to R2.** That is first-mover advantage
emerging endogenously from capacity commitment: the leader builds and locks in residual demand,
and the follower optimises against what is left.

Two consequences worth stating plainly.

**The equilibrium is not unique, so "the" answer to this game does not exist.** Reporting one
ordering would be reporting an artefact of the solution procedure. Sweep the order — and if you
extend this, sweep the starting profile too.

**No cycling appeared here, but do not generalise from that.** A fixed price makes each firm's
problem well behaved. Once price responds to total quantity (4c), a firm's optimal capacity depends
on the rival's output through the price, and 2-cycles become much more likely.

In [ ]:
res = iterate_best_response(first='R1', max_iter=10)
pd.DataFrame(res['log'])

The trace shows the mechanism directly: R1 moves first against an empty market and takes 1,652
units; R2 then optimises against the residual and takes 987. From iteration 1 onward neither firm
changes anything — a fixed point.


## 7. The cost of rivalry — and a bound that looks violated

Comparing 4a to 4b directly is invalid: 4a minimises **cost**, 4b maximises **profit**. The clean
approach is to take the competitive **decisions** and re-price them through the cooperative cost
accounting.

But there is a trap waiting here, and it is worth walking into deliberately, because the naive
version of this comparison produces a result that is **provably impossible**.

The planner has strictly more freedom than the firms: it can choose *any* pair of plans, including
exactly the competitive one. So the planner's cost must be a **lower bound** on the competitive cost.
If your comparison says otherwise, the comparison is wrong — not the theory.


In [ ]:
res = iterate_best_response(first='R1', max_iter=10)
comp_cost, comp_sales = 0.0, {}
for r in REGIONS:
    other = 'R2' if r == 'R1' else 'R1'
    mbr = best_response(r, res['sales'][other])
    comp_cost += mbr._h['cost'].getValue()
    for rt in REGIONS:
        for p in P:
            comp_sales[r, rt, p] = mbr._h['sale'][rt, p].X
served = {(rt, p): sum(comp_sales[r, rt, p] for r in REGIONS)
          for rt in REGIONS for p in P}

coop = solve_planner(0.5, learning='both')
coop_cost = sum(coop._H[r]['cost'].getValue() for r in REGIONS)

print("--- the NAIVE comparison ---")
print(f"  cooperative planner cost   {coop_cost:10.1f}")
print(f"  competitive cost           {comp_cost:10.1f}")
print(f"  apparent cost of rivalry   {comp_cost-coop_cost:10.1f}"
      f"  ({100*(comp_cost-coop_cost)/coop_cost:+.1f}%)")
print(f"\n  demand over horizon        {sum(DEMAND.values()):10.1f}")
print(f"  served by the firms        {sum(served.values()):10.1f}")
print(f"  UNSERVED                   {sum(DEMAND.values())-sum(served.values()):10.1f}")


The apparent cost of rivalry is **negative** — competition looks *cheaper* than the planner. That
cannot be an efficiency gain.

**The resolution is in the last line.** The planner is required to serve all demand (or pay a
shortfall penalty); the firms are not, and they simply decline to serve about 104 rate-units.
So the competitive outcome **is not in the planner's feasible set**, and the two numbers are not
comparable. Serving that last tranche of demand requires committing another lumpy facility, and the
planner pays for it while the firms walk away.

Two ways to repair the comparison. They answer different questions, and both restore the bound.


In [ ]:
# TEST 1 -- volume-matched: hold the planner to exactly what the firms chose to serve
mv = gp.Model(); mv.Params.OutputFlag = 0; mv.Params.MIPGap = 1e-6
Hv = {r: add_region(mv, r, 'both') for r in REGIONS}
mv.addConstrs((gp.quicksum(Hv[r]['sale'][rt, p] for r in REGIONS) >= served[rt, p]
               for rt in REGIONS for p in P), name='match_volume')
mv.setObjective(gp.quicksum(Hv[r]['cost'] for r in REGIONS), GRB.MINIMIZE)
mv.optimize()

# TEST 2 -- welfare-inclusive: charge the firms the same social cost of unserved demand
pen_comp = sum(OMEGA[p]*PEN_SHORT*max(0.0, DEMAND[rt, p]-served[rt, p])
               for rt in REGIONS for p in P)
coop_total = coop_cost + coop._pen.getValue()

print("TEST 1 -- volume-matched (efficiency only)")
print(f"  planner at competitive volume {mv.ObjVal:10.1f}")
print(f"  competitive                   {comp_cost:10.1f}")
print(f"  cost of rivalry               {comp_cost-mv.ObjVal:10.1f}"
      f"  ({100*(comp_cost-mv.ObjVal)/mv.ObjVal:+.2f}%)")
print(f"  bound holds? {mv.ObjVal <= comp_cost + 1e-3}")
print("\nTEST 2 -- welfare-inclusive (efficiency + unserved demand)")
print(f"  planner cost + penalty        {coop_total:10.1f}")
print(f"  competitive cost + penalty    {comp_cost+pen_comp:10.1f}")
print(f"  cost of rivalry               {comp_cost+pen_comp-coop_total:10.1f}"
      f"  ({100*(comp_cost+pen_comp-coop_total)/coop_total:+.1f}%)")
print(f"  bound holds? {coop_total <= comp_cost + pen_comp + 1e-3}")


Both repairs restore the bound, and the two numbers answer different questions:

| Comparison | Cost of rivalry | What it measures |
|---|---|---|
| Naive (different volumes) | **−9.5%** | nothing — invalid |
| Volume-matched | **+0.21%** | pure productive inefficiency of splitting output between two firms |
| Welfare-inclusive | **+33.6%** | inefficiency **plus** the social loss from 104 units never produced |

The volume-matched figure is small, which is itself informative: given *what* gets produced, two
firms produce it almost as cheaply as a planner would. Nearly all of the welfare cost of rivalry in
this model is the **under-provision** — capacity that no firm finds privately worth building.

That under-provision is an artefact of the **fixed price**, though. At a constant price the revenue
on a marginal unit cannot rise to justify a lumpy facility. With an endogenous price (Part 4c),
unserved demand raises the price and draws capacity in — so the quantity distortion largely
disappears and the remaining gap becomes interpretable as genuine strategic inefficiency.

**The general lesson**: when a comparison violates a bound you can prove, the error is in the
comparison. Check first whether the two models are solving the same problem — most often they differ
in a constraint (here, the obligation to serve demand) rather than in anything economic.


## 8. Learning under rivalry

Part 3b found production learning did not change the plan under cost minimization, because
cumulative production was pinned by demand. In the game, quantity is a **decision**, so the channel
finally has a lever. The effect should be small at this model size, but it should exist.

In [ ]:
rows = []
for lm in ['capacity', 'both']:
    res = iterate_best_response(learning=lm, first='R1', max_iter=10)
    last = {L['firm']: L for L in res['log'][-2:]}
    rows.append(dict(learning=lm, status=res['status'],
                     profit_R1=round(last['R1']['profit'], 1),
                     profit_R2=round(last['R2']['profit'], 1),
                     sales_R1=round(last['R1']['total_sales'], 1),
                     sales_R2=round(last['R2']['total_sales'], 1),
                     builds_R1=last['R1']['builds'], builds_R2=last['R2']['builds']))
pd.DataFrame(rows)

Adding the production channel raises both firms' profits — it is a cost reduction — while the
market split moves only slightly. The asymmetry in `EXPERIENCE0` is doing what it was built to do:
R1 starts several tiers up the curve and keeps a durable operating-cost advantage that R2's cheaper
capital cannot fully offset.

**Keep the channel in the formulation.** Its effect here is modest and partly obscured by the small
instance, but it is the mechanism that makes flooding rational in 4c, and the cost of carrying it is
only the tier binaries.

## 9. Summary and what comes next

### Findings

| Question | Answer |
|---|---|
| Is 4a a game? | **No.** One objective = one decision maker. It is a cooperative bound |
| What shape is the Pareto frontier? | Nearly bang-bang, because investment is lumpy |
| Does a pure-strategy equilibrium exist? | Yes in this instance — fixed point from both orders |
| Is the equilibrium unique? | **No.** First-mover advantage ≈ 29–34% of profit |
| Is rivalry more expensive? | +0.21% volume-matched, +33.6% welfare-inclusive. A naive comparison gives an impossible **negative** answer |
| Does production learning matter now? | Slightly — quantity is finally a decision |

### Formulation lessons

- **One `add_region` used three ways** — twice for the planner, once for a best response, later as
  the follower block in the bilevel model. Shared verified code across all of Part 4.
- **Vertical integration avoids transfer prices.** Cross-region sourcing between *rivals* would
  need an internal price at every stage; competing only downstream sidesteps it.
- **Distinguish fixed point from cycle.** A profile repeating immediately is convergence; repeating
  after $k \ge 2$ rounds means no pure-strategy equilibrium was found.
- **Sweep the move order.** With a non-unique equilibrium, one ordering is an artefact.
- **Never compare a cost objective to a profit objective.** Fix the competitive decisions and
  re-price them through the cooperative cost accounting.

### Next: 4c — Cournot with endogenous price

$$p_{rt,p} = a_{rt,p} - b \sum_{r} \text{sale}_{r,rt,p}$$

Each firm's revenue becomes quadratic but is concave in its own quantity, so each best response is
a **QP** — still tractable. Three things should change qualitatively:

1. The quantity distortion in §7 should largely vanish, making the cost-of-rivalry number
   interpretable.
2. **Flooding becomes rational**: extra output depresses the rival's margin *and* advances your own
   learning tier. This is where `PEN_DISPOSE` finally earns its calibration.
3. **Cycling becomes likely**, because each firm's optimum now depends on the rival's quantity
   through the price.

Then 4d (Stackelberg via the follower's KKT conditions — available only because the operational
layer is an LP) and 4e (tariffs, quotas, local content as exogenous swept levers).

### Things to try

- `PRICE_FIXED = 11` or `13` — at 10 no new capacity is built at all; at 13 both firms build out
  fully. The interesting band is narrow, which is itself informative
- `EXPERIENCE0 = {'R1': 500, 'R2': 500}` — remove the incumbency and watch the split symmetrise
- `TRANSPORT` cross-region down to 1.0 — markets integrate and rivalry intensifies
- `set_tier_minimums`-style local content applied to one region only, as a 4e preview